# NivoTime Benefits Intelligence Suite — AI/ML Simulation

This notebook reproduces the AI/ML layer described in Section 7 of the PRD
(`prd/src/prd_part3.md`) and Section 7 of the strategy report — **on synthetic data**,
not real NivoTime or client data (none was available for this project).

It builds a calibrated synthetic dataset of 600 employer groups (matched to published
aggregate statistics such as IRDAI's mean premium/life and typical claims ratios), then
trains and genuinely evaluates four of the five AI/ML models proposed in the report:

1. **Renewal-cost forecast** — Gradient Boosting Regressor, evaluated against a naive baseline.
2. **Peer-group matching** — k-Nearest Neighbours, for one sample client.
3. **Plan-design simulator** — log-linear rating relativities (what a plan change does to premium).
4. **Claims-anomaly detection** — Isolation Forest, plus a hybrid rule that raises precision/recall.

> **Every metric below is a real, computed result on synthetic data.** Nothing is invented.
> But a strong result here is *proof the mechanics work*, not proof the models will be this
> accurate on NivoTime's real book of business — that can only be shown by back-testing on
> real pilot data, as both documents recommend before any client-facing accuracy claim.

**Run all cells top to bottom** (Runtime → Run all). No local setup needed — this only uses
packages already installed in Colab.


## 0. Setup


In [ ]:
import numpy as np, pandas as pd, json
from sklearn.ensemble import GradientBoostingRegressor, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, precision_score, recall_score

rng = np.random.default_rng(42)
N = 600


## 1. Generate the synthetic employer-group dataset

600 employer groups with industry, group size, age profile, family definition, sum insured,
parents' co-pay, room-rent cap and city tier — with premium and claims-ratio built from
realistic rating relativities (see `prd/src/prd_part4.md`, Appendix A for the full generation spec).


In [ ]:
ind = rng.choice(["IT services", "Manufacturing", "BFSI", "Pharma", "Retail", "Prof. services"], N, p=[.28, .22, .16, .12, .12, .10])
age_base = {"IT services": 30, "Manufacturing": 37, "BFSI": 34, "Pharma": 35, "Retail": 31, "Prof. services": 33}
avg_age = np.array([age_base[i] for i in ind]) + rng.normal(0, 2.5, N)
emp = np.clip(np.exp(rng.normal(np.log(650), 0.6, N)), 250, 2000).astype(int)
family = rng.choice(["E", "E+S+2C", "E+S+2C+P"], N, p=[.15, .50, .35])
si = rng.choice([300000, 500000, 1000000], N, p=[.35, .45, .20])
copay_par = np.where(family == "E+S+2C+P", rng.choice([0, 10, 20], N, p=[.45, .35, .20]), 0)
room = rng.choice(["1%", "2%", "No cap"], N, p=[.45, .30, .25])
tier = rng.choice(["Metro", "Tier 2"], N, p=[.7, .3])

fam_f = {"E": 0.55, "E+S+2C": 1.0, "E+S+2C+P": 1.45}
si_f = {300000: 0.85, 500000: 1.0, 1000000: 1.22}
room_f = {"1%": 0.94, "2%": 1.0, "No cap": 1.07}
age_f = np.exp((avg_age - 33) * 0.035)
f_fam = np.array([fam_f[f] for f in family]); f_si = np.array([si_f[s] for s in si]); f_room = np.array([room_f[r] for r in room])
f_cop = 1 - copay_par / 100 * 0.55 * (family == "E+S+2C+P")
f_tier = np.where(tier == "Metro", 1.05, 0.92)
prem = 2105 * age_f * f_fam * f_si * f_room * f_cop * f_tier * np.exp(rng.normal(0, 0.08, N))

# claims ratio: parents cover w/o co-pay and no cap push it up; small groups more volatile
icr = (0.78 + 0.10 * (family == "E+S+2C+P") * (copay_par == 0) + 0.04 * (room == "No cap") + 0.004 * (avg_age - 33)
       + rng.normal(0, 0.12, N) * np.sqrt(650 / emp))
icr = np.clip(icr, 0.35, 1.6)
engagement = np.clip(rng.normal(0.44, 0.12, N), 0.1, 0.9)
trend = 0.12
cred = np.minimum(1, emp / 1000) ** 0.5
renewal = (trend + cred * 0.55 * (icr - 0.80) + 0.02 * (family == 'E+S+2C+P') + 0.015 * (room == 'No cap')
           + 0.004 * (avg_age - 33) + 0.01 * (tier == 'Metro') + rng.normal(0, 0.02, N))  # next-year loading

df = pd.DataFrame(dict(industry=ind, employees=emp, avg_age=avg_age.round(1), family=family, sum_insured=si,
                       copay_parents=copay_par, room_rent=room, city=tier, premium_per_life=prem.round(0),
                       claims_ratio=icr.round(3), engagement=engagement.round(3), renewal_loading=renewal.round(4)))
df.to_csv("synthetic_employer_groups.csv", index=False)
df.head()


## 2. Model: renewal-cost forecast (Gradient Boosting)

Predicts next year's renewal loading (% premium increase) from this year's group profile.
Compared against a naive baseline (always predict the market trend, 12%).


In [ ]:
X = pd.get_dummies(df.drop(columns=["renewal_loading", "engagement", "premium_per_life"]), drop_first=False).astype(float)
y = df["renewal_loading"]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=7)
gbm = GradientBoostingRegressor(n_estimators=300, max_depth=3, learning_rate=0.05, random_state=7).fit(Xtr, ytr)
pred = gbm.predict(Xte)
mae = mean_absolute_error(yte, pred) * 100
mae_naive = mean_absolute_error(yte, np.full(len(yte), trend)) * 100
within5 = float(np.mean(np.abs(yte - pred) <= 0.05) * 100)

print(f"Forecast MAE: {mae:.2f} pts   |   Naive-baseline MAE: {mae_naive:.2f} pts   |   Improvement: {(1 - mae/mae_naive)*100:.0f}%")
print(f"Share of test-set forecasts within +/-5 pts of the actual renewal: {within5:.1f}%")


In [ ]:
# Driver importance (permutation importance, grouped by feature family)
pi = permutation_importance(gbm, Xte, yte, n_repeats=10, random_state=7)
imp = pd.Series(pi.importances_mean, index=X.columns)
groups = {"Claims ratio (ICR)": ["claims_ratio"], "Family definition": [c for c in X if c.startswith("family")],
          "Average age": ["avg_age"], "Room-rent cap": [c for c in X if c.startswith("room")],
          "Parents' co-pay": ["copay_parents"], "Group size": ["employees"], "Industry": [c for c in X if c.startswith("industry")],
          "Sum insured": ["sum_insured"], "City tier": [c for c in X if c.startswith("city")]}
gimp = {k: float(imp[v].sum()) for k, v in groups.items()}
tot = sum(max(v, 0) for v in gimp.values())
gimp = {k: round(max(v, 0) / tot * 100, 1) for k, v in sorted(gimp.items(), key=lambda x: -x[1])}
gimp


## 3. Model: peer-group matching (k-Nearest Neighbours)

For one sample client (an IT-services group with the family-plus-parents definition and no
parents' co-pay), find its 42 nearest peers by industry, size, age, family definition, sum
insured and city, then benchmark it against them.


In [ ]:
feat = pd.get_dummies(df[["industry", "employees", "avg_age", "family", "sum_insured", "city"]], drop_first=False).astype(float)
Z = StandardScaler().fit_transform(feat)
client = int(df[(df.industry == "IT services") & (df.family == "E+S+2C+P") & (df.copay_parents == 0)].index[0])
nn = NearestNeighbors(n_neighbors=43).fit(Z)
_, idx = nn.kneighbors(Z[[client]])
peers = df.iloc[idx[0][1:]]
c = df.iloc[client]
pct = lambda col: round(float((peers[col] < c[col]).mean() * 100))
peer_view = dict(n_peers=len(peers), prem_percentile=pct("premium_per_life"),
                 icr_client_pct=round(float(c.claims_ratio) * 100), icr_peer_median_pct=round(float(peers.claims_ratio.median()) * 100),
                 engagement_client_pct=round(float(c.engagement) * 100), engagement_peer_median_pct=round(float(peers.engagement.median()) * 100),
                 forecast_renewal_pct=round(float(gbm.predict(X.iloc[[client]])[0]) * 100, 1))
peer_view


## 4. Model: plan-design simulator (log-linear rating relativities)

Fits premium as a log-linear function of plan features, then reads off the % premium
impact of specific plan changes a broker or HR team might simulate.


In [ ]:
Xr = pd.get_dummies(df[["avg_age", "family", "sum_insured", "copay_parents", "room_rent", "city"]].assign(
    sum_insured=df.sum_insured.astype(str)), drop_first=True).astype(float)
Xr["copay_parents"] = df.copay_parents * (df.family == "E+S+2C+P")
lr = LinearRegression().fit(Xr, np.log(df.premium_per_life))
coef = dict(zip(Xr.columns, lr.coef_))
sim = {
    "10% co-pay on parents": round((np.exp(coef["copay_parents"] * 10) - 1) * 100, 1),
    "20% co-pay on parents": round((np.exp(coef["copay_parents"] * 20) - 1) * 100, 1),
    "Room-rent cap 1% (from no cap)": round((np.exp(-coef.get("room_rent_No cap", 0) + coef.get("room_rent_2%", 0) * 0) - 1) * 100, 1),
    "Sum insured Rs 5L -> Rs 10L": round((np.exp(coef.get("sum_insured_1000000", 0) - coef.get("sum_insured_500000", 0)) - 1) * 100, 1),
    "Drop parents' cover": round((np.exp(coef.get("family_E+S+2C", 0) - coef.get("family_E+S+2C+P", 0)) - 1) * 100, 1),
}
r2 = lr.score(Xr, np.log(df.premium_per_life))
print(f"Model fit: R^2 = {r2:.3f}")
sim


## 5. Model: claims-anomaly detection (Isolation Forest + hybrid rule)

5,000 synthetic claims with 2% injected anomalies (inflated amount, compressed length of
stay). A pure Isolation Forest is compared against a hybrid: Isolation Forest plus a
cost-per-day threshold rule — which the PRD documents as the recommended approach, since
the pure model alone under-performs.


In [ ]:
M = 5000
amt = np.exp(rng.normal(np.log(45000), 0.7, M)); los = np.clip(rng.poisson(3, M), 1, 20).astype(float)
age = rng.integers(1, 80, M).astype(float)
lab = np.zeros(M, int); k = int(M * 0.02); an = rng.choice(M, k, replace=False)
amt[an] *= rng.uniform(3, 6, k); los[an] = np.clip(los[an] * rng.uniform(0.2, 0.5, k), 1, 20); lab[an] = 1
C = np.c_[np.log(amt), los, np.log(amt / los)]
iso = IsolationForest(contamination=0.02, random_state=7).fit(C)
flag = (iso.predict(C) == -1).astype(int)
iso2 = IsolationForest(contamination=0.05, random_state=7).fit(C)
cpd = amt / los
hyb = ((iso2.predict(C) == -1) & (cpd > np.percentile(cpd, 95))).astype(int)
anom = dict(claims=M, injected=k, flagged=int(flag.sum()), precision=round(precision_score(lab, flag) * 100),
            recall=round(recall_score(lab, flag) * 100), hyb_flagged=int(hyb.sum()),
            hyb_precision=round(precision_score(lab, hyb) * 100), hyb_recall=round(recall_score(lab, hyb) * 100))
print(f"Isolation Forest alone:  precision {anom['precision']}%  recall {anom['recall']}%")
print(f"Hybrid (+ cost-per-day rule): precision {anom['hyb_precision']}%  recall {anom['hyb_recall']}%")
print("-> Human review is still required before any claim is challenged (AI-governance principle, report Sec. 12.4).")


## 6. Save all results

Writes the same `sim.json` that `prd/src/prd_part3.md` (Section 7) and the dashboard
mockups (`prd/src/prd_figs.py`) were built from — so you can confirm every number in the
PRD traces back to this notebook.


In [ ]:
out = dict(n_groups=N, mean_premium=round(float(df.premium_per_life.mean())), mean_icr=round(float(df.claims_ratio.mean()) * 100, 1),
           forecast=dict(mae_pp=round(mae, 2), naive_mae_pp=round(mae_naive, 2), within_5pp=round(within5, 1), test_n=len(yte),
                         improvement=round((1 - mae / mae_naive) * 100)),
           drivers=gimp, peer=peer_view, simulator=sim, sim_r2=round(r2, 3), anomaly=anom,
           industry_counts=df.industry.value_counts().to_dict())
json.dump(out, open("sim.json", "w"), indent=1, default=str)
print(json.dumps(out, indent=1, default=str))


## 7. (Optional) Push your changes back to GitHub from Colab

If you've cloned this repo into Colab (or mounted it via Google Drive) and want to push a
change — e.g. a re-run with different assumptions — from right here:

```python
# 1) One-time: clone with a Personal Access Token (create one at github.com/settings/tokens,
#    scope 'repo'). Never paste the token in a cell you'll share — use Colab's Secrets (key
#    icon in the left sidebar) or getpass() so it isn't saved in the notebook.
from getpass import getpass
token = getpass('GitHub token: ')
!git clone https://{token}@github.com/<your-username>/<your-repo>.git repo
%cd repo
```

```python
# 2) After editing / re-running:
!git config user.email "anvesharajsingh@gmail.com"
!git config user.name "Anvesha"
!git add -A
!git commit -m "Re-run AI/ML simulation from Colab"
!git push
```

Alternatively, use Colab's built-in **File → Save a copy in GitHub** menu, which handles
authentication through a Google/GitHub OAuth popup instead of a token.
